Week 4: Baseline Model

This notebook takes the processed data from the 02_preprocessing.ipynb file and focuses on modeling. A baseline Linear Regression model was developed to predict residential ClosePrice. The dataset was divided into training and validation sets, and the model was trained using numerical, categorical, and binary features. The model's performance was evaluated using $R^2$, Mean Absolute Error(MAE), Root Mean Squared Error(RMSE), Median Absolute Error, and Median Absolute Percentage Error(MDAPE). These metrics were reported for both training and validation sets to assess how well the model performed on the data it learned and the unseen data.  

In [119]:
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import category_encoders as ce
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error


In [120]:
path_dir = Path("raw")
files = sorted(path_dir.glob("CRMLSSold*.csv"))
train_files = files[:-1]
test_file = files[-1]

train_list = [pd.read_csv(file, low_memory=False) for file in train_files]
df = pd.concat(train_list, ignore_index=True)

test_df = pd.read_csv(test_file, low_memory=False)

print(f"Training Period: {train_files[0].name} to {train_files[-1].name}") 
print(f"Test file: {test_file.name}")

print("The combined shape of Train DataFrame is:", df.shape)
print("Test shape:", test_df.shape)


Training Period: CRMLSSold202502.csv to CRMLSSold202607.csv
Test file: CRMLSSold202608.csv
The combined shape of Train DataFrame is: (400050, 80)
Test shape: (23131, 79)


In [121]:
df_scoped = df[
        (df['PropertyType'] == 'Residential') & 
        (df['PropertySubType'] == 'SingleFamilyResidence')].copy()
print("The shape of scoped DataFrame is:", df_scoped.shape)


The shape of scoped DataFrame is: (201413, 80)


Data Preprocessing:

1. Remove exact duplicate rows
2. Remove rows with missing close dates or close dates later than the selected cutoff date.
3. Remove latitude and longitude values outside the defined geographic boundaries.
4. Missing values in GarageSpaces were filled with 0, assuming that a missing value indicates the property has no garage.
5. Rows containing zero were dropped from ClosePrice, LivingArea, BedroomsTotal, BathroomsTotalInteger, Latitude, and Longitude.
6. Rows containing null were dropped from ClosePrice, LivingArea, BedroomsTotal, BathroomsTotalInteger, Latitude, Longitude, PostalCode, and City
7. Replace zero values in LotSizeSquareFeet with NaN which will be handled later using median imputation.

In [122]:
# Determine the percentage of null values for each feature.
null_rates = (df_scoped.isnull().mean()*100).round(2)
print(null_rates[null_rates > 0].sort_values(ascending=False))

TaxYear                   100.00
BusinessType              100.00
FireplacesTotal           100.00
AboveGradeFinishedArea    100.00
TaxAnnualAmount           100.00
                           ...  
ListAgentFullName           0.03
ListAgentAOR                0.02
BathroomsTotalInteger       0.01
ListAgentLastName           0.01
PurchaseContractDate        0.01
Length: 62, dtype: float64


In [123]:
rows_before = len(df_scoped)
df_scoped['CloseDate'] = pd.to_datetime(df_scoped['CloseDate'].astype(str), errors='coerce')
cutoff_date = pd.Timestamp('2026-07-31')

# Count rows where CloseDate is null or greater than cutoff_date
df_scoped = df_scoped[
    df_scoped['CloseDate'].notna() & (df_scoped['CloseDate'] <= cutoff_date)
    ].copy()

# Find rows where all the values are duplicate
all_dup_count = df_scoped.duplicated().sum()
print(f"Exact duplicate rows (entire row matches): {all_dup_count}")

df_scoped = df_scoped.drop_duplicates(keep='first')
rows_removed = rows_before - len(df_scoped)
print(f"Total rows removed: {rows_removed}")

Exact duplicate rows (entire row matches): 6
Total rows removed: 6


In [124]:
numeric_features = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "ClosePrice", 
            "Latitude", "Longitude","LotSizeSquareFeet", "YearBuilt",
            "Stories", "GarageSpaces"]

categorical_features = ["PostalCode", "City"] 
                        
binary_features = ["NewConstructionYN", "PoolPrivateYN", "ViewYN", 
            "FireplaceYN", "CountyOrParish"]

features = numeric_features + categorical_features + binary_features

for col in numeric_features:
    if col in df_scoped.columns:
        df_scoped[col] = pd.to_numeric(df_scoped[col], errors='coerce')

# Define valid coordinate ranges for California
lat_min, lat_max = 32.0, 42.0
lon_min, lon_max = -124.0, -114.0

# Identify rows with invalid or out-of-bounds coordinates
invalid_lat = (df_scoped['Latitude'] < lat_min) | (df_scoped['Latitude'] > lat_max)
invalid_lon = (df_scoped['Longitude'] < lon_min) | (df_scoped['Longitude'] > lon_max)
invalid_location_count = (invalid_lat | invalid_lon).sum()

print(f"Rows with out-of-bounds coordinates: {invalid_location_count}")

# Drop rows outside these bounds
df_scoped = df_scoped[~(invalid_lat | invalid_lon)].copy()
rows_removed = rows_before - (len(df_scoped))
pct_removed = (rows_removed / rows_before) * 100 if rows_before > 0 else 0

print(f"Total rows before: {rows_before}")
print(f"Total rows removed: {rows_removed} ({pct_removed:.2f}%)")

Rows with out-of-bounds coordinates: 61
Total rows before: 201413
Total rows removed: 67 (0.03%)


Handling Missing Values
1. Null GarageSpaces probably means no Garage so fill it with 0.
2. Drop zero and missing values. 

In [125]:
df_model = df_scoped[features].copy()
before = len(df_model)

# Drop ClosePrice and LivingArea 0 and less
df_model = df_model[
            (df_model["ClosePrice"] > 0) & (df_model["LivingArea"] > 0) &
            (df_model["BedroomsTotal"] > 0) & (df_model["BathroomsTotalInteger"] > 0) &
            (df_model["Latitude"] != 0) & (df_model["Longitude"] != 0) & 
            (df_model["YearBuilt"] >= 1900)
             ].copy()

# Drop null values
df_model = df_model.dropna(subset=["ClosePrice", "BedroomsTotal", "BathroomsTotalInteger",
                                   "LivingArea", "Latitude", "Longitude", "City", "PostalCode", "YearBuilt"]).copy()

# Fill null garagespaces with 0
df_model["GarageSpaces"] = df_model["GarageSpaces"].fillna(0)

print(f'After Filtering:{df_model.shape}')

print(f"Removed {before - len(df_model)} rows "
    f"({(before-len(df_model)) / before * 100: .2f}%)")


After Filtering:(199998, 17)
Removed 1348 rows ( 0.67%)


In [126]:
for col in binary_features:
    if col in df_model.columns:
        df_model[col] = (
            df_model[col]
            .map({True: "True", False: "False"}).fillna("Unknown").astype(str))
print("All binary feature values are now strings!")

All binary feature values are now strings!


In [127]:
cols = ["ClosePrice", "LivingArea", "BedroomsTotal", 
        "BathroomsTotalInteger", "LotSizeSquareFeet"]

percentiles = [0.0001, 0.005, 0.995, 0.9999]

# Calculate actual values at each percentile
quantile_values = df_model[cols].quantile(percentiles).T

# Rename columns to show percentile labels
quantile_values.columns = [
    f"{p * 100:g}th percentile" for p in percentiles
]

print(quantile_values.round(2).to_string())

                       0.01th percentile  0.5th percentile  99.5th percentile  99.99th percentile
ClosePrice                       31000.0          190000.0         8700000.00        5.500070e+07
LivingArea                         375.0             661.0            6769.01        1.639002e+04
BedroomsTotal                        1.0               1.0               6.00        1.200000e+01
BathroomsTotalInteger                1.0               1.0               7.00        1.400000e+01
LotSizeSquareFeet                    0.0            1306.0          483378.36        8.265302e+08


Removing Extreme Outliers from Training and Validation Datasets
1. ClosePrice, LivingArea, and LotSizeSquareFeet weres filtered using 0.01st and 99.99th percentile.
2. For BedroomTotal and BathroomsTotalInteger only upper threshold were applied since the zero were already removed in the previous step. 

In [128]:
before = len(df_model)

both_cols = ["ClosePrice", "LivingArea", "LotSizeSquareFeet"]
upper_only_cols = ["BedroomsTotal", "BathroomsTotalInteger"]

lower = df_model[both_cols].quantile(0.0001)
upper = df_model[both_cols + upper_only_cols].quantile(0.9999)

keep_mask = (
    df_model[both_cols].ge(lower) &
    df_model[both_cols].le(upper[both_cols])).all(axis=1)

keep_mask &= df_model[upper_only_cols].le(upper[upper_only_cols]).all(axis=1)

removed = (~keep_mask).sum()
removed_pct = removed / before * 100 if before else 0

df_model = df_model.loc[keep_mask].copy()

print(f"Total rows before: {before:,}")
print(f"Total rows removed: {removed:,}")
print(f"Percentage removed: {removed_pct:.2f}%")
print(f"Total rows remaining: {len(df_model):,}")

Total rows before: 199,998
Total rows removed: 3,532
Percentage removed: 1.77%
Total rows remaining: 196,466


In [129]:
df = df_model.copy()
X = df.drop(columns = ["ClosePrice"])
y = df["ClosePrice"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)


In [130]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)


Numeric features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'Latitude', 'Longitude', 'LotSizeSquareFeet', 'YearBuilt', 'Stories', 'GarageSpaces']
Categorical features: ['PostalCode', 'City', 'NewConstructionYN', 'PoolPrivateYN', 'ViewYN', 'FireplaceYN', 'CountyOrParish']


In [131]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())])

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

train_rmse = np.sqrt( mean_squared_error(y_train, y_train_pred))

val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

val_mae = mean_absolute_error(y_val, y_val_pred)

def mdape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    nonzero = y_true != 0
    if not np.any(nonzero):
        return np.nan

    percentage_errors = (np.abs(y_true[nonzero] - y_pred[nonzero]) / np.abs(y_true[nonzero])) * 100
    return np.median(percentage_errors)

def regression_report(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred) * 100,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedianAbsoluteError": median_absolute_error(y_true, y_pred),
        "MDAPE": mdape(y_true, y_pred)
    }

train_report = regression_report(y_train, y_train_pred)
val_report = regression_report(y_val, y_val_pred)

print("\nTRAINING REPORT")
print(f"R2: {train_report['R2']:.2f}%")
print(f"MAE: ${train_report['MAE']:,.2f}")
print(f"RMSE: ${train_report['RMSE']:,.2f}")
print(f"Median Absolute Error: ${train_report['MedianAbsoluteError']:,.2f}")
print(f"MDAPE: {train_report['MDAPE']:.2f}%")

print("\nVALIDATION REPORT")
print(f"R2: {val_report['R2']:.2f}%")
print(f"MAE: ${val_report['MAE']:,.2f}")
print(f"RMSE: ${val_report['RMSE']:,.2f}")
print(f"Median Absolute Error: ${val_report['MedianAbsoluteError']:,.2f}")
print(f"MDAPE: {val_report['MDAPE']:.2f}%")



TRAINING REPORT
R2: 73.08%
MAE: $311,503.17
RMSE: $702,839.47
Median Absolute Error: $193,236.18
MDAPE: 20.26%

VALIDATION REPORT
R2: 71.01%
MAE: $317,599.30
RMSE: $746,636.91
Median Absolute Error: $199,133.07
MDAPE: 20.66%


Results:
1. The model achieved an $R^2$ of 73.08% on training data and 71.01% on validation data, showing that the model can account for a large portion of the differences in prices.
2. The small difference between training and validation results suggests the model generalizes well to new data.
3. The average MAE is large meaning that predictions can differ significantly from actual prices.
4. Large RMSE indicates that some predictions have much larger errors than others.

Thus, the model shows promise but needs improvement to make accurate price predictions. 

In [132]:
diagnostics = pd.DataFrame({
    "actual": np.asarray(y_val).ravel(),
    "original_pred": np.asarray(y_val_pred).ravel()})

# Evaluate the original model by target range
diagnostics["price_group"] = pd.cut(
    diagnostics["actual"],
    bins=[
        -np.inf,
        1000000,
        3000000,
        10000000,
        50000000,
        np.inf],
    labels=[
        "Under $1M",
        "$1M-$3M",
        "$3M-$10M",
        "$10M-$50M",
        "Over $50M"])

print("\nOriginal model")

for group, data in diagnostics.groupby("price_group", observed=True):
    actual = data["actual"].to_numpy()
    pred = data["original_pred"].to_numpy()

    print(f"\n{group} (n={len(data)})")
    print("MAE:", round(mean_absolute_error(actual, pred), 2))
    print("Median absolute error:", round(median_absolute_error(actual, pred), 2))
    print("RMSE:", round(np.sqrt(mean_squared_error(actual, pred)), 2))

    nonzero = actual != 0
    if nonzero.any():
        mdape_value = np.median(np.abs(actual[nonzero] - pred[nonzero]) / np.abs(actual[nonzero]) * 100)
        print("MDAPE:", round(mdape_value, 2), "%")

# Count extreme observations in training and validation
print("\nEXTREME VALUE COUNTS")

for name, y in [
    ("Training", y_train),("Validation", y_val)]:
    y = np.asarray(y).ravel()

    print(f"\n{name}:")
    print("Over $10M:", np.sum(y > 10000000))
    print("Over $50M:", np.sum(y > 50000000))
    print("Over $100M:", np.sum(y > 100000000))
    print("Over $500M:", np.sum(y > 500000000))
    


Original model

Under $1M (n=22594)
MAE: 216657.29
Median absolute error: 174106.37
RMSE: 290933.59
MDAPE: 27.15 %

$1M-$3M (n=14499)
MAE: 322300.88
Median absolute error: 219566.5
RMSE: 503372.92
MDAPE: 14.28 %

$3M-$10M (n=2063)
MAE: 944647.25
Median absolute error: 676074.1
RMSE: 1307067.91
MDAPE: 16.18 %

$10M-$50M (n=136)
MAE: 6442927.09
Median absolute error: 5029959.97
RMSE: 8138477.03
MDAPE: 39.88 %

Over $50M (n=2)
MAE: 43252987.78
Median absolute error: 43252987.78
RMSE: 43511201.69
MDAPE: 82.49 %

EXTREME VALUE COUNTS

Training:
Over $10M: 546
Over $50M: 1
Over $100M: 0
Over $500M: 0

Validation:
Over $10M: 138
Over $50M: 2
Over $100M: 0
Over $500M: 0


In [133]:
errors = pd.DataFrame({
    "Actual": np.asarray(y_val).ravel(),
    "Predicted": np.asarray(y_val_pred).ravel()})

errors["AbsoluteError"] = (
    errors["Actual"] - errors["Predicted"]).abs()

errors["AbsolutePercentageError"] = np.where(
    errors["Actual"] != 0,
    errors["AbsoluteError"]/ np.abs(errors["Actual"]) * 100, np.nan)

print("\nLARGEST VALIDATION ERRORS")

print(errors.sort_values(
        "AbsoluteError", ascending=False).head(20).to_string(index=False))


LARGEST VALIDATION ERRORS
    Actual    Predicted  AbsoluteError  AbsolutePercentageError
53000000.0 5.013755e+06   4.798624e+07                90.540084
51750000.0 1.323027e+07   3.851973e+07                74.434263
47000000.0 1.172587e+07   3.527413e+07                75.051333
29000000.0 6.304059e+06   2.269594e+07                78.261867
23700000.0 1.986641e+06   2.171336e+07                91.617550
29500000.0 1.154685e+07   1.795315e+07                60.858128
29000000.0 1.109697e+07   1.790303e+07                61.734602
24000000.0 6.834169e+06   1.716583e+07                71.524294
22750000.0 6.429413e+06   1.632059e+07                71.738843
26055000.0 9.997121e+06   1.605788e+07                61.630700
21775000.0 6.051958e+06   1.572304e+07                72.206853
26000000.0 1.069615e+07   1.530385e+07                58.860964
19750000.0 4.715569e+06   1.503443e+07                76.123699
23000000.0 8.926310e+06   1.407369e+07                61.189957
23000000.0 9.

Next week, I plan to test Decision Tree and Random Forest to see they can improve prediction accuray and reduce errors.